In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Recommender-system-related implementations
from src.models import ItemMeanRecommender, BiasBaselineRecommender   # baselines
from src.models import UserKNNBasicRecommender                        # main model
from src.models import KNNBaselineRecommender                         # improved model

from src.data_splits import load_premade_splits
from src.evaluator import RecommenderEvaluator
from src.debug import low_coverage_check

# 1. Data

In [2]:
df = pd.read_csv(
    'data/u.data', 
    sep='\t', 
    encoding='utf-8', 
    header=None, 
    names=['user_id', 'item_id', 'rating', 'timestamp']
)
df.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [10]:
df.sort_values('timestamp', ascending=True, inplace=True)

for fold, (train_df, test_df) in enumerate(load_premade_splits("data"), start=1):

    # =================== training ===================
    model_knn = UserKNNBasicRecommender(n_neighbors=50, metric='cosine', center_ratings=False)
    model_knn.fit(train_df)

    print('='*75, '\n', "Fold: ", fold, sep='')
    # ================= sanity check =================
    # print("sanity check:\n", '-'*25, sep='')
    # low_coverage_check(train_df, test_df, model_knn)
    # ================== evaluation ==================
    # print("\nevaluation metrics:\n", '-'*25, sep='')
    evaluator = RecommenderEvaluator(model_knn, test_df, relevance_threshold=4, prec_rec_k=10)
    # eval_results = evaluator.evaluation_report()

Fold: 1
user_id
1      79
2      12
3       5
4       8
5      25
       ..
455     2
456     1
457     7
458     3
462     1
Name: item_id, Length: 456, dtype: int64
Fold: 2
user_id
1      29
2      14
3       6
4       4
5      18
       ..
651     1
653     3
654     4
655     2
658     1
Name: item_id, Length: 644, dtype: int64
Fold: 3
user_id
1      28
2       9
3       1
4       2
5       8
       ..
868     1
870     1
871     1
875     2
876     1
Name: item_id, Length: 849, dtype: int64
Fold: 4
user_id
1      13
2       2
3       1
4       3
5       5
       ..
939    11
940    20
941     7
942    26
943    32
Name: item_id, Length: 890, dtype: int64
Fold: 5
user_id
1      14
2       3
3       2
4       2
5       2
       ..
939    28
940    37
941    11
942    40
943    64
Name: item_id, Length: 878, dtype: int64


In [4]:
# ================== prediction & interpretation ==================
ratings_pred = model_knn.predict_ratings(test_df)
test_user_id = np.random.choice(test_df['user_id'].unique())
top_n_recs = model_knn.recommend_items(test_user_id, n=10)
print(f"Top-10 recommendations for user {test_user_id}: {top_n_recs}")

# TODO: When running this code, notice how the first similarity is almost 0. Investigate why this is and whether it's a problem.
neighbors = model_knn.get_neighbors(test_user_id)
print(f"Users most similar to user {test_user_id}:")
for neighbor, similarity in neighbors:
    print(f"    - User {neighbor} (similarity: {similarity:.3f})")

Top-10 recommendations for user 497: [(np.int64(887), np.float64(5.050232933494736)), (np.int64(909), np.float64(5.050232933494736)), (np.int64(733), np.float64(5.050232933494736)), (np.int64(900), np.float64(5.050232933494736)), (np.int64(848), np.float64(5.050232933494736)), (np.int64(814), np.float64(5.050232933494736)), (np.int64(896), np.float64(5.050232933494736)), (np.int64(886), np.float64(5.050232933494736)), (np.int64(865), np.float64(5.050232933494736)), (np.int64(656), np.float64(5.050232933494736))]
Users most similar to user 497:
    - User 222 (similarity: 0.532)
    - User 276 (similarity: 0.517)
    - User 417 (similarity: 0.516)
    - User 435 (similarity: 0.513)
    - User 56 (similarity: 0.501)
    - User 268 (similarity: 0.499)
    - User 303 (similarity: 0.486)
    - User 472 (similarity: 0.485)
    - User 102 (similarity: 0.484)
    - User 109 (similarity: 0.481)
    - User 727 (similarity: 0.464)
    - User 682 (similarity: 0.463)
    - User 178 (similarity: 0.4

d:\Quang\2026_Spring_AIL303m\Capstone1\src\models\knn_cf_basic.py:140: RuntimeWarning: invalid value encountered in divide
  scores = self.user_means[u] + numerator / denom


In [5]:
# TODO: Evaluation. Compare the recommendations generated by your KNN collaborative filtering model with a simple baseline (e.g., recommending the most popular items) using appropriate evaluation metrics (e.g., precision, recall, F1-score).
# TODO: Analyze a few limitations of the KNN approach and implement one improvement to address one of the limitations.
    # Experiment 1: Vs. centering ratings
    # Experiment 2: Vs. KNNBaseline (bias-based KNN)
# TODO: Write report